# ⚡ The Flux Capacitor — *hosted edition*
## What *is* a Large Language Model?

**Section 2** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### ☁️ This is the **hosted** version of Demo 1.
> It needs an API key but **no GPU**. If you have a graphics card and want to
> see more, open **`01_flux_capacitor_local.ipynb`** instead — it can show you
> something this notebook genuinely cannot, and we will be honest about that
> when we get there.

---

Everybody has now seen a chatbot type an answer back at them. Almost nobody has
been shown **what is actually happening** while it does that.

It is not looking anything up. There is no sentence stored anywhere inside it
waiting to be found. It is doing one thing, very fast, over and over:

> ### 🎯 Guessing the next small piece of text.

```mermaid
flowchart LR
    A["Your text"] --> B["Split into<br/>tokens"]
    B --> C["Score every<br/>possible next token"]
    C --> D["Pick one"]
    D --> E["Add it to<br/>the text"]
    E --> F{"Finished?"}
    F -- "no" --> C
    F -- "yes" --> G["Done"]
```

---

### What you need

| | |
|---|---|
| ⏱️ | About 3 minutes |
| 🔑 | An API key in your `.env` file |
| 🎮 | No GPU required |
| 💵 | A few cents at most |

---

# 🚦 LIVE DEMO PATH

## Setup

Two things: a client for the hosted model, and a **tokenizer** so we can look at
how text gets chopped up. The tokenizer is a few megabytes — no model weights,
no GPU.

In [1]:
from delorean import llm, viz

client = llm.get_api_client()
MODEL = llm.api_model()

# A tokenizer only — a few MB, not gigabytes of model weights.
tokenizer = llm.load_tokenizer("tiny")

print(f"Talking to : {MODEL}")
print("Tokenizer  : Qwen3.5 (for illustration — see the note in Act 1)")

Talking to : claude-haiku-4-5
Tokenizer  : Qwen3.5 (for illustration — see the note in Act 1)


---

## Act 1 — 🔢 Words become numbers

Before the model sees your question, your question stops being words.

It gets chopped into **tokens** — pieces roughly the size of a syllable — and
each piece becomes a number. The model only ever sees numbers.

> 📌 **A necessary caveat.** Every model family has its own tokenizer, and
> Anthropic does not publish theirs. The chips below come from Qwen's tokenizer,
> which we can inspect. The *idea* is identical for every model on the market;
> only the exact split differs. Further down we ask the hosted model for its own
> real token count.

In [2]:
viz.show_tokens(tokenizer, "appraisal")
viz.show_tokens(tokenizer, "parcel 04-127-3891")

> ### ⚠️ Great Scott — look at that parcel number.
>
> **Thirteen tokens.** Every digit is its own separate piece. The model does not
> see `04-127-3891` as an identifier — it sees thirteen unrelated fragments.
>
> **Remember this.** It is why language models fumble parcel numbers, account
> numbers and arithmetic, and we come back to it in Section 4.

Now the real count, from the hosted model itself. This is a free API call —
it costs nothing and generates nothing. 💸

In [3]:
sample = "The homestead exemption reduces a homeowner's property tax."

counted = client.messages.count_tokens(
    model=MODEL,
    messages=[{"role": "user", "content": sample}],
)

print(f'"{sample}"')
print(f"\n  words                     : {len(sample.split())}")
print(f"  tokens, per {MODEL:<14}: {counted.input_tokens}")

"The homestead exemption reduces a homeowner's property tax."

  words                     : 8
  tokens, per claude-haiku-4-5: 20


> ## 💡 The model has never seen a word. It sees numbers.

---

## Act 2 — 🔒 The part this notebook cannot show you

In the local version of this notebook, Act 2 is the whole point of the demo.

We reach inside the model mid-sentence and pull out **the actual odds it assigns
to every possible next token** — all 248,320 of them — and draw them as a bar
chart. You watch it be 99% certain about one thing and genuinely torn about
another. It looks like this:

```
An appraiser estimates the fair market ▮

   ·value  ████████████████████████████  99.4%
   ·price  ▏                              0.2%
   ·sale   ▏                              0.0%
```

**You cannot do that here, and neither can anyone else.**

Those raw scores are called *logits*, and **no major hosted provider returns
them** — not Anthropic, not OpenAI, not Google. You send text in and you get
text back. The reasoning is a sealed box.

> ### 🔑 This is not a complaint. It is the argument.
>
> There is a real, permanent difference between *using* a model and *running*
> one:
>
> | | Hosted ☁️ | Local 🖥️ |
> |---|---|---|
> | Quality today | Excellent | Good and improving fast |
> | Setup | An API key | A graphics card |
> | Your data | Leaves the building | Never leaves |
> | Cost shape | Per question, forever | Hardware once |
> | **Can you see inside?** | **No** | **Yes** |
>
> For most office work the hosted models are the pragmatic choice. But when the
> question is *"why did it produce that number?"* — the one question an appraisal
> office will always eventually be asked — only one of these columns can answer.
>
> **Section 9 comes back to this** with models small enough to run on the machine
> under your desk.

If you have a GPU, stop here and open **`01_flux_capacitor_local.ipynb`**. It is
worth the three minutes. 🖥️

---

## Act 3 — 🔁 Do it again. And again. And again.

If the model only predicts *one* token, how do you get a paragraph?

You take the winner, glue it onto the end of the text, and **ask the same
question again**. Then again, until it decides to stop. That loop is generation.

Watch it arrive. 👀

In [4]:
canvas = viz.TokenCanvas(title=f"{MODEL}, streaming over the network", boundaries=False)

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content":
               "In one sentence, what does a property appraiser do?"}],
) as stream:
    for chunk in stream.text_stream:
        canvas.add(chunk)

> 📌 **One more honest caveat.** Those blocks are *network chunks*, not tokens.
> The API sends text in whatever pieces are convenient, and does not tell you
> where one token ended and the next began. The local notebook shows you the
> real boundaries — this is the same limitation as Act 2, wearing a different hat.

> ## 💡 Everything you have ever seen ChatGPT do is this loop, running fast.

---

## Act 4 — 🏘️ Now ask it something real

Same loop, a real question from our own work.

In [5]:
canvas = viz.TokenCanvas(title=MODEL, boundaries=False)

with client.messages.stream(
    model=MODEL,
    max_tokens=700,
    messages=[{"role": "user", "content":
               "What are the three approaches to value in real property "
               "appraisal? Answer briefly, for an audience of assessors."}],
) as stream:
    for chunk in stream.text_stream:
        canvas.add(chunk)

---

## 🎬 That's Section 2

A language model is a **next-token guesser** — a very sophisticated one, trained
on an enormous amount of text, but a guesser.

Three things to carry forward:

1. 🔢 **It sees tokens, not words** — which is why identifiers and arithmetic are
   a weak spot *(Section 4)*
2. 🎲 **Every answer is a chain of bets** — "confident" does not mean "correct"
3. 🔒 **A hosted model is a sealed box** — you can use it, but you cannot look
   inside it *(Section 9)*

And notice what it *didn't* do. It didn't know who it was, or that it was
talking to an appraiser. It had no instructions at all.

> ### ⏭️ **Next: Section 3 — Setting the Time Circuits.**
> Punch a destination into the dashboard and the same box behaves completely
> differently.

---

---

# 🔬 EXPLORE ON YOUR OWN

*Off the live path — for when you take this back to the office.*

---

## 5. 💵 What does this actually cost?

The honest answer for most offices is *less than you think, until it isn't*.
Cost scales with every question, forever — which is the trade against buying a
graphics card once.

In [6]:
# USD per MILLION tokens, as of this writing. Always check current pricing.
PRICES = {
    "claude-opus-5":    (5.00, 25.00),   # most capable
    "claude-sonnet-5":  (3.00, 15.00),   # balanced
    "claude-haiku-4-5": (1.00,  5.00),   # cheapest
}

question = "Summarize the three approaches to value for a property appraiser."
n_in = client.messages.count_tokens(
    model=MODEL, messages=[{"role": "user", "content": question}]
).input_tokens
n_out = 400   # a generous estimate for the answer

print(f"One question ({n_in} tokens in, ~{n_out} out):\n")
for name, (p_in, p_out) in PRICES.items():
    cost = (n_in / 1e6) * p_in + (n_out / 1e6) * p_out
    per_day = cost * 500     # a busy office asking 500 questions a day
    print(f"  {name:18} ${cost:.5f} per question   ~${per_day:6.2f}/day at 500 questions")

One question (22 tokens in, ~400 out):

  claude-opus-5      $0.01011 per question   ~$  5.06/day at 500 questions
  claude-sonnet-5    $0.00607 per question   ~$  3.03/day at 500 questions
  claude-haiku-4-5   $0.00202 per question   ~$  1.01/day at 500 questions


> 💡 Change `ANTHROPIC_MODEL` in your `.env` to try a cheaper one. For most of
> the demos in this repository, the cheapest model is perfectly adequate.

---

## 6. ✏️ Ask your own question

Change the text and run it.

In [7]:
my_question = "Explain the difference between market value and assessed value."

canvas = viz.TokenCanvas(title=MODEL, boundaries=False)
with client.messages.stream(
    model=MODEL,
    max_tokens=600,
    messages=[{"role": "user", "content": my_question}],
) as stream:
    for chunk in stream.text_stream:
        canvas.add(chunk)

---

## 7. 🖥️ What you could run yourself

These are the models the local notebooks use — all of them free, all of them
running entirely on your own hardware.

In [8]:
from delorean import models

models.compare()


Qwen3.5 0.8B  (0.8B, fp16)
  Qwen/Qwen3.5-0.8B
  License : Apache 2.0
  VRAM    : 1.44 GB measured   Speed: ~14 tokens/sec
  Good at :
      + Starting up in about three seconds
      + Running on a laptop with no GPU at all
      + Showing how generation works, one token at a time
  Watch   :
      - Not for agent work — it is here to be understood, not to reason
      - Will confidently get facts wrong

Qwen3.5 4B  (4B, 4-bit NF4)
  Qwen/Qwen3.5-4B
  License : Apache 2.0
  VRAM    : 2.98 GB measured   Speed: ~14 tokens/sec
  Good at :
      + Tool calling — ranked 1st of 13 local models in a 2026 evaluation
      + Long documents (262,144 token context)
      + Punching far above its size; it beat models 5-7x larger
  Watch   :
      - Eager to call several tools at once when one at a time would do
      - Thinking mode is ON by default and adds a pause before answering

SmolLM3 3B  (3B, 4-bit NF4)
  HuggingFaceTB/SmolLM3-3B
  License : Apache 2.0
  VRAM    : not measured here
  Goo

---

<div align="center">

### 🕰️ *"Your future hasn't been written yet. No one's has.* 🕰️
### *Your future is whatever you make it. So make it a good one."*

🚗💨 ⚡ 🏘️

</div>